In [3]:

# Install required libraries
!pip install --upgrade pip
!pip install "wandb>=0.15"
!pip install transformers==4.51.3
!pip install datasets evaluate rouge_score einops timm peft
!pip install "accelerate>=0.26.0"
!pip install python-dotenv
!pip install optuna

In [4]:
import os
import math
import optuna
import wandb
from optuna.pruners import SuccessiveHalvingPruner
from optuna.exceptions import TrialPruned
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    Trainer,
    TrainingArguments,
    TrainerCallback
)
from peft import get_peft_model, LoraConfig, TaskType
import torch
import numpy as np
import evaluate
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import EarlyStoppingCallback

In [5]:
load_dotenv("secrets.env")
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

MODEL_NAME = "microsoft/Florence-2-base-ft"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Load full dataset
raw_ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")['raw']

# Load metrics
bleu, meteor, rouge = map(evaluate.load, ["bleu", "meteor", "rouge"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
<unknown>:515: SyntaxWarning: invalid escape sequence '\d'


Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import pandas as pd
from collections import Counter
print(raw_ds.features)
print(f"total: {len(raw_ds)}")
print(f"{list(raw_ds.features.keys())}")
answer_cnt = Counter(raw_ds["answer"])
print(f"answer: {len(answer_cnt)}")
print(f"sourc: {len(set(raw_ds['source']))}")
print(f"question: {len(set(raw_ds['question']))}")
print(f"img_id: {len(set(raw_ds['img_id']))}")


In [7]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits[0], axis=-1)
  pred_texts = processor.tokenizer.batch_decode(preds,skip_special_tokens=True)
  label_ids = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
  label_texts = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
  pred_texts = [p.strip() for p in pred_texts]
  label_texts = [l.strip() for l in label_texts]
  return {"rougeL": rouge.compute(predictions=pred_texts, references=label_texts)["rougeL"]}

In [8]:
class OptunaPruningCallback(TrainerCallback):
    def __init__(self, trial, metric_name: str):
        self.trial = trial
        self.metric_name = metric_name

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        score = metrics.get(self.metric_name)
        step = state.global_step
        self.trial.report(score, step)
        if self.trial.should_prune():
            raise TrialPruned(f"Trial was pruned at step {step}")
        return control

In [9]:
def sample_hyperparams(trial):
  lr       = trial.suggest_loguniform("learning_rate", 1e-6, 1e-4)
  train_bs = trial.suggest_categorical("per_device_train_batch_size", [2, 4])
  grad_acc = trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4])
  weight_decay = trial.suggest_uniform("weight_decay", 0.0, 0.1)
  lora_r      = trial.suggest_categorical("lora_r", [4, 8, 16])
  lora_alpha  = trial.suggest_categorical("lora_alpha", [8, 16, 32])
  lora_dropout= trial.suggest_uniform("lora_dropout", 0.0, 0.3)

  return {
      "lr": lr,
      "train_bs": train_bs,
      "grad_acc": grad_acc,
      "weight_decay": weight_decay,
      "lora_r": lora_r,
      "lora_alpha": lora_alpha,
      "lora_dropout": lora_dropout,
  }

In [10]:
# -- 3) Prepare small train/val split (1.5% coverage) --
from datasets import ClassLabel
def create_small_dataset(raw_ds, coverage=0.015, seed=7, test_size=0.1):

  if coverage < 1:
      dataset = raw_ds.train_test_split(
          test_size=coverage,
          seed=seed,
          shuffle=True
      )
      dataset = dataset["test"].train_test_split(test_size=test_size, seed=seed)

      train_dataset, val_dataset = dataset['train'], dataset['test']

  else:
      dataset = dataset.train_test_split(test_size=test_size, seed=seed)
      train_dataset, val_dataset = dataset['train'], dataset['test']
  return train_dataset, val_dataset

In [11]:
def build_model_with_lora(MODEL_NAME, params):
  model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
  peft_config = LoraConfig(
      r=params["lora_r"],
      lora_alpha=params["lora_alpha"],
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "lm_head", "fc2"],
      task_type="CAUSAL_LM",
      lora_dropout=params["lora_dropout"],
      bias="none",
      inference_mode=False
  )
  model = get_peft_model(model, peft_config)
  return model.to(DEVICE)

In [12]:
def create_collate_fn(processor):
    def collate_fn(batch):
        questions = [x["question"] for x in batch]
        images = [x["image"].convert("RGB") for x in batch]
        answers = [x["answer"] for x in batch]

        inputs = processor(text=questions, images=images, return_tensors="pt", padding=True)
        labels = processor.tokenizer(answers, return_tensors="pt", padding=True).input_ids
        labels[labels == processor.tokenizer.pad_token_id] = -100
        inputs["labels"] = labels
        return inputs
    return collate_fn

In [13]:
def create_training_args(run_name, params, max_steps, steps_per_epoch):
    return TrainingArguments(
        output_dir=run_name,
        per_device_train_batch_size=params["train_bs"],
        gradient_accumulation_steps=params["grad_acc"],
        per_device_eval_batch_size=params["train_bs"],
        max_steps=max_steps,
        learning_rate=params["lr"],
        weight_decay=params["weight_decay"],
        logging_steps=10,
        save_strategy="no",
        eval_strategy="steps",
        eval_steps=steps_per_epoch,
        report_to=["wandb"],
        metric_for_best_model="rougeL",
        greater_is_better=True,
        fp16=True,
        remove_unused_columns=False,
    )

In [14]:
def objective(trial):
    run_name = f"hparam_trial_{trial.number}"
    wandb.init(project="florence2_hparam_trial", name=run_name, reinit=True)

    # 1. 取超参
    params = sample_hyperparams(trial)
    wandb.config.update(params)

    # 2. 划分数据
    train_ds, val_ds = create_small_dataset(raw_ds)

    # 3. 模型
    model = build_model_with_lora(MODEL_NAME, params)

    # 4. 数据处理
    collate_fn = create_collate_fn(processor)

    # 5. 训练步数
    effective_bs = params["train_bs"] * params["grad_acc"]
    steps_per_epoch = math.ceil(len(train_ds) / effective_bs)
    max_steps = steps_per_epoch * 1

    # 6. 训练参数
    training_args = create_training_args(run_name, params, max_steps, steps_per_epoch)

    # 7. 训练器
    pruning_cb = OptunaPruningCallback(trial, metric_name="eval_rougeL")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        callbacks=[pruning_cb, EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # 8. 训练 & 评估
    trainer.train()
    results = trainer.evaluate()
    wandb.finish()

    return results["eval_rougeL"]

In [ ]:
pruner = SuccessiveHalvingPruner()
study = optuna.create_study(
    direction="maximize",
    pruner=pruner,
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=50)

print("Best trial:")
print(study.best_trial)

[I 2026-04-10 06:52:51,260] A new study created in memory with name: no-name-2f2d64b5-70ad-45f1-8e11-dc231d14f28f
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: lea0128z (lea0128z-linkedin) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/tmp/ipykernel_16896/1400181902.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr       = trial.suggest_loguniform("learning_rate", 1e-6, 1e-4)
/tmp/ipykernel_16896/1400181902.py:5: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  weight_decay = trial.suggest_uniform("weight_decay", 0.0, 0.1)
/tmp/ipykernel_16896/1400181902.py:8: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  lora_dropout= trial.suggest_uniform("lora_dropout", 0.0, 0.3)
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embe

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
